# Text Data Analysis: 2D Parameter Matrix

This notebook analyzes parameter combinations to find which features affect image scores.
Creates matrices showing how different parameter pairs correlate with scores.

In [1]:
import sys
import os
from pathlib import Path

# Setup path
root_path = str(Path(os.getcwd()).parents[2])
sys.path.insert(0, root_path)

print(f"Root path: {root_path}")
print(f"Python path configured")

Root path: e:\ComfyUI\custom_nodes\comfyui-image-scorer
Python path configured


In [2]:
from shared.paths import text_data_file, scores_file, vectors_file
from shared.io import load_single_jsonl
from shared.matrix_analysis import MatrixAnalyzer
import json

# Load data
print("Loading data...")
text_data = load_single_jsonl(text_data_file)
scores = load_single_jsonl(scores_file)

print(f"Text data records: {len(text_data)}")
print(f"Score records: {len(scores)}")

if len(text_data) != len(scores):
    min_len = min(len(text_data), len(scores))
    print(f"WARNING: Data length mismatch! Using {min_len} records")
    text_data = text_data[:min_len]
    scores = scores[:min_len]
    
# Display sample
if text_data:
    print(f"\nSample text record keys: {list(text_data[0].keys())}")
    print(f"Sample score: {scores[0]}")


Loading data...
Text data records: 24446
Score records: 24446

Sample text record keys: ['lora', 'sampler', 'scheduler', 'model', 'steps', 'clip_skip', 'original_width', 'original_height', 'final_width', 'final_height', 'cfg', 'original_aspect_ratio', 'final_aspect_ratio', 'sharpness', 'noise_score', 'contrast', 'colorfulness', 'artifact_score', 'edge_density', 'texture_lbp', 'lora_weight', 'negative_prompt', 'positive_prompt']
Sample score: 1.0


In [3]:
# Initialize analyzer
# memory_limit controls max unique parameters to track
MEMORY_LIMIT = 10000  # Adjust based on RAM available

print("Initializing Matrix Analyzer...")
analyzer = MatrixAnalyzer(scores, text_data, memory_limit=MEMORY_LIMIT)

print(f"Matrix Analyzer initialized")
print(f"Memory limit: {MEMORY_LIMIT} unique parameters")
print(f"Ready to build comprehensive parameter matrix")

Initializing Matrix Analyzer...
Matrix Analyzer initialized
Memory limit: 10000 unique parameters
Ready to build comprehensive parameter matrix


## 2D Matrix Analysis

Analyze how combinations of parameters affect scores. Each cell shows score statistics for that parameter combination.

In [4]:
# Build comprehensive matrix combining ALL parameters
print("\n" + "="*80)
print("BUILDING COMPREHENSIVE 2D PARAMETER MATRIX")
print("="*80)
print("This creates a symmetric matrix of all parameter values from text data")
print("Each cell contains scores for records that have both parameters\n")

try:
    analyzer.build_matrix()
    print(f"✓ Matrix building complete")
except Exception as e:
    print(f"Error during matrix building: {e}")
    import traceback
    traceback.print_exc()


BUILDING COMPREHENSIVE 2D PARAMETER MATRIX
This creates a symmetric matrix of all parameter values from text data
Each cell contains scores for records that have both parameters

Processing 24446 text records...


Extracting parameters: 100%|██████████| 24446/24446 [00:02<00:00, 8313.76 records/s]


Found 5344 unique parameters
Building parameter co-occurrence matrix...


Building matrix: 100%|██████████| 24446/24446 [00:48<00:00, 500.21 records/s]

Matrix built: 5344x5344 parameters
✓ Matrix building complete


In [5]:
# Calculate statistics for all matrix cells
print("\n" + "="*80)
print("CALCULATING STATISTICS FOR MATRIX CELLS")
print("="*80 + "\n")

try:
    stats: dict[tuple[int, int], dict[str, float]] = analyzer.calculate_statistics(min_count=1000)
    print(f"✓ Statistics calculated for {len(stats)} cells")
except Exception as e:
    print(f"Error during statistics calculation: {e}")
    import traceback
    traceback.print_exc()


CALCULATING STATISTICS FOR MATRIX CELLS



Flattening Matrix: 100%|██████████| 14281840/14281840 [00:17<00:00, 830985.60cells/s] 
e:\ComfyUI\custom_nodes\comfyui-image-scorer\shared\matrix_analysis.py:255: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df = pl.DataFrame(


Stats: Kept 18898 cells, Dropped 14262942 cells (Min Count: 1000)
Calculating Statistics via Polars (Multithreaded)...


Building Final Dict: 100%|██████████| 18898/18898 [00:00<00:00, 432352.00it/s]


✓ Statistics calculated for 37329 cells


In [6]:
# Display matrix summary
print("\n" + "="*80)
print("MATRIX SUMMARY")
print("="*80)

try:
    summary = analyzer.get_matrix_summary()
    for key, value in summary.items():
        print(f"  {key}: {value}")
        
    size = analyzer.get_matrix_size()
    print(f"\n  Matrix Size: {size[0]}x{size[1]} parameters")
except Exception as e:
    print(f"Error getting summary: {e}")
    import traceback
    traceback.print_exc()


MATRIX SUMMARY
  total_parameters: 5344
  matrix_cells: 37329
  total_score_entries: 86508061.0
  mean_of_means: 2.9342522715368458
  loaded_records: 24446
  loaded_vectors: 24446

  Matrix Size: 5344x5344 parameters


## Export Results

Save all matrices to JSON for further analysis

In [7]:
# Export matrix to JSON
print("\n" + "="*80)
print("EXPORTING MATRIX ANALYSIS")
print("="*80 + "\n")

output_path = Path("matrix_analysis_results.json")

try:
    analyzer.export_to_json(str(output_path))
    print(f"✓ Matrix exported to: {output_path}")
    print(f"✓ Total cells with data: {len(analyzer.cell_stats)}")
except Exception as e:
    print(f"Error during export: {e}")
    import traceback
    traceback.print_exc()


EXPORTING MATRIX ANALYSIS



Exporting to JSON: 100%|██████████| 37329/37329 [00:00<00:00, 2470714.44 cells/s]


✓ Exported 18898 cell statistics to matrix_analysis_results.json
✓ Matrix exported to: matrix_analysis_results.json
✓ Total cells with data: 37329
